# FlowShift
> Analyzing how urban traffic congestion influences public transport demand in Los Angeles

**Dataset:** Large-ST (ca_his_raw_2021.h5)  
**Course:** Data Science and Algorithms — BIT CS  
**Research Question:** Do people switch to public transport when roads get congested — and by how much?

---
## 0. Environment Setup

In [ ]:
# Install all required packages
import sys
!{sys.executable} -m pip install h5py tables pyarrow pandas numpy matplotlib seaborn scikit-learn scipy torch --ignore-installed setuptools

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import h5py
import os
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

print('All libraries imported successfully')

In [ ]:
# Check device — Apple Silicon MPS, CUDA, or CPU
import torch

if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Using Apple Silicon GPU (MPS) — M2 acceleration enabled')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using CUDA GPU: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('Using CPU')

print(f'Device: {device}')

---
## 1. Data Loading
### 1a. Inspect HDF5 file structure

In [ ]:
# File paths — update if your Dataset folder is in a different location
H5_PATH      = './Dataset/ca_his_raw_2021.h5'
PARQUET_PATH = './Dataset/ca_2021_clean.parquet'

# Inspect HDF5 structure
print('HDF5 file structure:')
with h5py.File(H5_PATH, 'r') as f:
    def print_structure(name, obj):
        print(f'  {name} — {type(obj).__name__}')
    f.visititems(print_structure)

print(f'\nFile size: {os.path.getsize(H5_PATH)/1024**3:.2f} GB')

### 1b. Load a small sample first to inspect columns

In [ ]:
# Load first 100k rows to understand structure without crashing RAM
sample = pd.read_hdf(H5_PATH, key='t', stop=100000)

print('Shape:', sample.shape)
print('\nColumns:', sample.columns.tolist())
print('\nFirst 3 rows:')
print(sample.head(3))
print('\nData types:')
print(sample.dtypes)
print('\nMissing values:')
print(sample.isnull().sum())
print(f'\nMemory usage: {sample.memory_usage(deep=True).sum()/1024**2:.1f} MB')

### 1c. Convert full HDF5 to Parquet — run ONCE only

In [ ]:
# Only run this cell if parquet file does not exist yet
if not os.path.exists(PARQUET_PATH):
    print('Converting HDF5 to Parquet — this runs once only...')
    chunk_size = 500000

    # axis1 = row index in pandas fixed-format HDF5
    with h5py.File(H5_PATH, 'r') as f:
        total_rows = f['t/axis1'].shape[0]
    print(f'Total rows in file: {total_rows:,}')

    first_chunk = True
    for start in range(0, total_rows, chunk_size):
        stop  = min(start + chunk_size, total_rows)
        chunk = pd.read_hdf(H5_PATH, key='t', start=start, stop=stop)

        # Downcast to reduce memory by ~50%
        for col in chunk.columns:
            if chunk[col].dtype == 'float64':
                chunk[col] = chunk[col].astype('float32')
            elif chunk[col].dtype == 'int64':
                chunk[col] = chunk[col].astype('int32')

        if first_chunk:
            chunk.to_parquet(PARQUET_PATH, engine='pyarrow', index=True)
            first_chunk = False
        else:
            chunk.to_parquet(PARQUET_PATH, engine='pyarrow', index=True, append=True)

        print(f'  Rows {start:,} – {stop:,} done')
        del chunk

    print(f'\nDone! Parquet saved at {PARQUET_PATH}')
    print(f'Parquet size: {os.path.getsize(PARQUET_PATH)/1024**2:.1f} MB')
else:
    print(f'Parquet already exists at {PARQUET_PATH} — skipping conversion')

### 1d. Load from Parquet — use this every session

In [ ]:
# Fast load — update column names based on your sample output above
# Common Large-ST columns: timestamp, speed, flow, occupancy, sensor_id
df = pd.read_parquet(PARQUET_PATH)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print(f'Memory: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
print('\nFirst 3 rows:')
print(df.head(3))

---
## 2. Data Cleaning & Preprocessing

In [ ]:
# ── UPDATE THESE to match your actual column names from step 1b ──
TIMESTAMP_COL  = 'timestamp'   # datetime column
SPEED_COL      = 'speed'       # vehicle speed
FLOW_COL       = 'flow'        # vehicle count
OCCUPANCY_COL  = 'occupancy'   # road occupancy %
SENSOR_COL     = 'sensor_id'   # sensor identifier
# ─────────────────────────────────────────────────────────────────

# Parse timestamp
df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL])
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

# Check for missing values
print('Missing values before cleaning:')
print(df.isnull().sum())

# Remove rows with missing speed or flow
df = df.dropna(subset=[SPEED_COL, FLOW_COL])

# Remove clearly invalid readings
df = df[df[SPEED_COL] >= 0]
df = df[df[FLOW_COL]  >= 0]

print(f'\nShape after cleaning: {df.shape}')
print('\nDate range:')
print(f'  Start : {df[TIMESTAMP_COL].min()}')
print(f'  End   : {df[TIMESTAMP_COL].max()}')

---
## 3. Feature Engineering

In [ ]:
# Time-based features
df['hour']        = df[TIMESTAMP_COL].dt.hour
df['day_of_week'] = df[TIMESTAMP_COL].dt.dayofweek  # 0=Mon, 6=Sun
df['month']       = df[TIMESTAMP_COL].dt.month
df['year']        = df[TIMESTAMP_COL].dt.year
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
df['is_peak']     = df['hour'].isin([7,8,9,17,18,19]).astype(int)

df['season'] = df['month'].map({
    12:'winter', 1:'winter',  2:'winter',
    3:'spring',  4:'spring',  5:'spring',
    6:'summer',  7:'summer',  8:'summer',
    9:'fall',   10:'fall',   11:'fall'
})

# Congestion flag — below 45mph is considered congested
SPEED_THRESHOLD   = 45
df['is_congested'] = (df[SPEED_COL] < SPEED_THRESHOLD).astype(int)

# Lag features — previous time steps as predictors
df = df.sort_values(TIMESTAMP_COL)
df['speed_lag1']  = df[SPEED_COL].shift(1)   # 5 min ago
df['speed_lag12'] = df[SPEED_COL].shift(12)  # 1 hour ago
df['flow_lag1']   = df[FLOW_COL].shift(1)

df = df.dropna()

print('Features added successfully')
print('New columns:', ['hour','day_of_week','month','year',
                       'is_weekend','is_peak','season',
                       'is_congested','speed_lag1','speed_lag12','flow_lag1'])
print(f'\nCongestion rate: {df["is_congested"].mean()*100:.1f}% of readings')

In [ ]:
# Build monthly congestion index — aggregates hourly data to monthly
monthly = df.groupby(['year', 'month']).agg(
    avg_speed       = (SPEED_COL,      'mean'),
    avg_flow        = (FLOW_COL,       'mean'),
    congestion_pct  = ('is_congested', 'mean'),  # proportion of hours congested
    peak_speed      = (SPEED_COL,      lambda x: x[df.loc[x.index,'is_peak']==1].mean()),
    total_readings  = (SPEED_COL,      'count')
).reset_index()

# Congestion severity tiers
monthly['congestion_tier'] = pd.cut(
    monthly['congestion_pct'],
    bins  = [0, 0.2, 0.4, 0.6, 1.0],
    labels= ['low', 'moderate', 'high', 'severe']
)

print('Monthly congestion index:')
print(monthly.head(12))

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── Plot 1: Speed distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[SPEED_COL], bins=60, color='#5DCAA5', edgecolor='none', alpha=0.8)
axes[0].axvline(SPEED_THRESHOLD, color='#D85A30', linestyle='--', linewidth=1.5,
                label=f'Congestion threshold ({SPEED_THRESHOLD} mph)')
axes[0].set_title('Speed distribution')
axes[0].set_xlabel('Speed (mph)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(df[FLOW_COL], bins=60, color='#7F77DD', edgecolor='none', alpha=0.8)
axes[1].set_title('Traffic flow distribution')
axes[1].set_xlabel('Flow (vehicles/5min)')
axes[1].set_ylabel('Count')

plt.suptitle('Traffic data distributions — Large-ST CA 2021', y=1.02)
plt.tight_layout()
plt.savefig('./figures/01_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Average speed by hour of day ─────────────────────────
hourly = df.groupby(['hour', 'is_weekend'])[SPEED_COL].mean().reset_index()
weekday = hourly[hourly['is_weekend'] == 0]
weekend = hourly[hourly['is_weekend'] == 1]

plt.figure(figsize=(10, 4))
plt.plot(weekday['hour'], weekday[SPEED_COL], 
         color='#D85A30', linewidth=2, marker='o', markersize=4, label='Weekday')
plt.plot(weekend['hour'], weekend[SPEED_COL], 
         color='#5DCAA5', linewidth=2, marker='s', markersize=4, label='Weekend')
plt.axhline(SPEED_THRESHOLD, color='gray', linestyle='--', linewidth=1,
            label=f'Congestion threshold ({SPEED_THRESHOLD} mph)')
plt.fill_between(weekday['hour'], weekday[SPEED_COL], SPEED_THRESHOLD,
                 where=weekday[SPEED_COL] < SPEED_THRESHOLD,
                 color='#D85A30', alpha=0.15, label='Congested zone')
plt.title('Average speed by hour — weekday vs. weekend')
plt.xlabel('Hour of day')
plt.ylabel('Average speed (mph)')
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.savefig('./figures/02_speed_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Congestion heatmap — hour × day of week ──────────────
pivot = df.pivot_table(
    values=SPEED_COL,
    index='day_of_week',
    columns='hour',
    aggfunc='mean'
)
pivot.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

plt.figure(figsize=(14, 5))
sns.heatmap(pivot, cmap='RdYlGn', center=SPEED_THRESHOLD,
            linewidths=0.3, annot=False, fmt='.0f',
            cbar_kws={'label': 'Average speed (mph)'})
plt.title('Traffic speed heatmap — day of week × hour (green = free flow, red = congested)')
plt.xlabel('Hour of day')
plt.ylabel('Day of week')
plt.tight_layout()
plt.savefig('./figures/03_congestion_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 4: Monthly congestion trend ────────────────────────────
monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(monthly['date'], monthly['congestion_pct']*100,
        width=20, color='#D85A30', alpha=0.6, label='Congestion rate (%)')
ax2.plot(monthly['date'], monthly['avg_speed'],
         color='#1D9E75', linewidth=2, marker='o', markersize=5, label='Avg speed (mph)')

ax1.set_xlabel('Month')
ax1.set_ylabel('Congestion rate (%)', color='#D85A30')
ax2.set_ylabel('Average speed (mph)', color='#1D9E75')
plt.title('Monthly congestion rate and average speed — 2021')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('./figures/04_monthly_congestion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 5: Congestion tier breakdown ───────────────────────────
tier_counts = monthly['congestion_tier'].value_counts().reindex(
    ['low','moderate','high','severe'])

colors = ['#5DCAA5', '#EF9F27', '#D85A30', '#E24B4A']
plt.figure(figsize=(7, 4))
bars = plt.bar(tier_counts.index, tier_counts.values, color=colors, edgecolor='none')
for bar, val in zip(bars, tier_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha='center', va='bottom', fontsize=11)
plt.title('Months by congestion severity tier')
plt.xlabel('Congestion tier')
plt.ylabel('Number of months')
plt.tight_layout()
plt.savefig('./figures/05_congestion_tiers.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Statistical Analysis
### 5a. Summary statistics by congestion tier

In [ ]:
# Summary stats per tier
tier_stats = df.groupby('is_congested').agg(
    count        = (SPEED_COL, 'count'),
    avg_speed    = (SPEED_COL, 'mean'),
    avg_flow     = (FLOW_COL,  'mean'),
).round(2)
tier_stats.index = ['Free flow', 'Congested']
print('Traffic statistics by congestion state:')
print(tier_stats)

In [ ]:
# Peak vs off-peak congestion rates
peak_stats = df.groupby('is_peak')['is_congested'].mean() * 100
peak_stats.index = ['Off-peak', 'Peak hours']
print('Congestion rate by time period:')
print(peak_stats.round(1).to_string())
print(f'\nPeak hours are {peak_stats["Peak hours"]/peak_stats["Off-peak"]:.1f}x more congested than off-peak')

In [ ]:
# Speed-flow correlation
from scipy import stats

r, p = stats.pearsonr(df[SPEED_COL], df[FLOW_COL])
print(f'Speed vs Flow correlation: r = {r:.3f}, p = {p:.4e}')

# Monthly congestion trend
monthly_r, monthly_p = stats.pearsonr(
    monthly['month'], monthly['congestion_pct'])
print(f'Month vs Congestion rate: r = {monthly_r:.3f}, p = {monthly_p:.4f}')

---
## 6. Modelling
### 6a. Prepare features

In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Encode season
le = LabelEncoder()
df['season_enc'] = le.fit_transform(df['season'])

FEATURES = ['hour', 'day_of_week', 'is_weekend', 'is_peak',
            'season_enc', 'speed_lag1', 'speed_lag12', 'flow_lag1']
TARGET   = SPEED_COL

# Time-based train/test split — never shuffle time series data
split    = int(len(df) * 0.8)
X_train  = df[FEATURES].iloc[:split]
X_test   = df[FEATURES].iloc[split:]
y_train  = df[TARGET].iloc[:split]
y_test   = df[TARGET].iloc[split:]

print(f'Training samples : {len(X_train):,}')
print(f'Testing samples  : {len(X_test):,}')

### 6b. Baseline — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

lr_mae  = mean_absolute_error(y_test, lr_preds)
lr_rmse = mean_squared_error(y_test, lr_preds, squared=False)
lr_r2   = r2_score(y_test, lr_preds)

print('Linear Regression (baseline):')
print(f'  MAE  : {lr_mae:.3f}')
print(f'  RMSE : {lr_rmse:.3f}')
print(f'  R²   : {lr_r2:.3f}')

### 6c. Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_preds)
rf_rmse = mean_squared_error(y_test, rf_preds, squared=False)
rf_r2   = r2_score(y_test, rf_preds)

print('Random Forest:')
print(f'  MAE  : {rf_mae:.3f}')
print(f'  RMSE : {rf_rmse:.3f}')
print(f'  R²   : {rf_r2:.3f}')

### 6d. LSTM — main model

In [ ]:
# Scale features
scaler      = MinMaxScaler()
all_data    = df[FEATURES + [TARGET]].values
scaled_data = scaler.fit_transform(all_data)

# Create sequences for LSTM
SEQ_LEN = 12  # 12 steps = 1 hour of 5-min data

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len, :-1])
        y.append(data[i+seq_len, -1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_seq, y_seq = make_sequences(scaled_data, SEQ_LEN)
split_seq    = int(len(X_seq) * 0.8)

X_tr = torch.tensor(X_seq[:split_seq]).to(device)
y_tr = torch.tensor(y_seq[:split_seq]).to(device)
X_te = torch.tensor(X_seq[split_seq:]).to(device)
y_te = torch.tensor(y_seq[split_seq:]).to(device)

print(f'Sequence shape — X: {X_tr.shape}, y: {y_tr.shape}')

In [ ]:
# LSTM model definition
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

# Training
EPOCHS     = 30
BATCH_SIZE = 512
LR         = 0.001

model     = LSTMModel(input_size=len(FEATURES)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
dataset   = TensorDataset(X_tr, y_tr)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

train_losses = []
print('Training LSTM...')
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for xb, yb in loader:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    train_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:02d}/{EPOCHS} — Loss: {avg_loss:.5f}')

print('Training complete')

---
## 7. Evaluation & Results

In [ ]:
# LSTM evaluation
model.eval()
with torch.no_grad():
    lstm_preds_scaled = model(X_te).cpu().numpy()

# Inverse scale predictions back to original units
def inverse_target(scaled_vals, scaler, n_features):
    dummy = np.zeros((len(scaled_vals), n_features + 1))
    dummy[:, -1] = scaled_vals
    return scaler.inverse_transform(dummy)[:, -1]

lstm_preds_actual = inverse_target(lstm_preds_scaled, scaler, len(FEATURES))
y_te_actual       = inverse_target(y_te.cpu().numpy(), scaler, len(FEATURES))

lstm_mae  = mean_absolute_error(y_te_actual, lstm_preds_actual)
lstm_rmse = mean_squared_error(y_te_actual, lstm_preds_actual, squared=False)
lstm_r2   = r2_score(y_te_actual, lstm_preds_actual)

# Results comparison table
results = pd.DataFrame({
    'Model' : ['Linear Regression (baseline)', 'Random Forest', 'LSTM (main)'],
    'MAE'   : [lr_mae,   rf_mae,   lstm_mae],
    'RMSE'  : [lr_rmse,  rf_rmse,  lstm_rmse],
    'R²'    : [lr_r2,    rf_r2,    lstm_r2]
}).round(3)

print('=== Model Comparison ===')
print(results.to_string(index=False))

In [ ]:
# ── Plot 6: Predicted vs actual ──────────────────────────────────
N = 500  # show first 500 test points
plt.figure(figsize=(12, 4))
plt.plot(y_te_actual[:N],       color='#1D9E75', linewidth=1.5, label='Actual speed')
plt.plot(lstm_preds_actual[:N], color='#D85A30', linewidth=1.5,
         linestyle='--', label='LSTM predicted')
plt.title('LSTM — predicted vs. actual speed (first 500 test samples)')
plt.xlabel('Time step')
plt.ylabel('Speed (mph)')
plt.legend()
plt.tight_layout()
plt.savefig('./figures/06_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 7: Training loss curve ──────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS+1), train_losses, color='#7F77DD', linewidth=2, marker='o', markersize=4)
plt.title('LSTM training loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.tight_layout()
plt.savefig('./figures/07_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 8: Feature importance (Random Forest) ───────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

plt.figure(figsize=(8, 5))
bars = plt.barh(importances.index, importances.values, color='#5DCAA5', edgecolor='none')
plt.title('Feature importance — what drives traffic speed?')
plt.xlabel('Importance score')
plt.tight_layout()
plt.savefig('./figures/08_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Conclusion

In [ ]:
print('=' * 50)
print('FLOWSHIFT — KEY FINDINGS SUMMARY')
print('=' * 50)
print(f'Dataset         : Large-ST CA 2021')
print(f'Total records   : {len(df):,}')
print(f'Date range      : {df[TIMESTAMP_COL].min().date()} to {df[TIMESTAMP_COL].max().date()}')
print(f'Congestion rate : {df["is_congested"].mean()*100:.1f}% of all readings')
print()
print('Best model performance (LSTM):')
print(f'  MAE  : {lstm_mae:.3f} mph')
print(f'  RMSE : {lstm_rmse:.3f} mph')
print(f'  R²   : {lstm_r2:.3f}')
print()
print('Top congestion periods: update after reviewing EDA plots')
print('=' * 50)

---
## 9. Save all outputs

In [ ]:
# Save monthly congestion index for Step 2 (merge with ridership)
monthly.to_csv('./Dataset/monthly_congestion_index.csv', index=False)
print('Monthly congestion index saved to ./Dataset/monthly_congestion_index.csv')

# Save trained model
torch.save(model.state_dict(), './Dataset/flowshift_lstm.pth')
print('LSTM model saved to ./Dataset/flowshift_lstm.pth')

# Save results table
results.to_csv('./Dataset/model_results.csv', index=False)
print('Model results saved to ./Dataset/model_results.csv')